In [1]:
import os
import sys

module_path = os.path.abspath('..')
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from torch.utils.data import DataLoader
from utils import args_transformer, args_gan
from datasets import GenDataset
from models import ConditionalDiffusionModel, LightningModelDiffusion

In [2]:
parser = args_gan()
args = parser.parse_args([
    "--img_size", "5",
    "--metadata_path", "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/metadata.pkl",
    "--dataset_path", "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/{}/{}/{}/{}.zip",
    "--gan_ind_path", "/pscratch/sd/b/botaoli/SFGD_VA/Data/NN_Data_compressed/gan_ind.pkl",
    "--label_size", "7",
    "--particle", "proton_contained",
])

train_set = GenDataset(args, split="train")
train_loader = DataLoader(
    train_set, collate_fn=train_set.collate_fn, batch_size=args.batch_size,
    num_workers=args.num_workers, shuffle=True)

In [3]:
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

checkpoint_path = "/pscratch/sd/b/botaoli/SFGD_VA/Results/cnf/test_diffusion/checkpoints/proton_contained/train_loss/last-v1.ckpt"

model = ConditionalDiffusionModel(
    time_dim=128,
    cond_dim=128,
    base_channels=32,
    num_timesteps=1000,
    img_size=args.img_size,
)

lit_model = LightningModelDiffusion.load_from_checkpoint(
    checkpoint_path,
    diffusion_model=model,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
lit_model = lit_model.to(device)
lit_model.eval()

# Example image
event = train_set[1231]
x_ini, y_ini, z_ini = event['pos_ini']
ke = event['ke']
dx, dy, dz = event['dir_ini']
kin = torch.tensor([[x_ini, y_ini, z_ini, ke, dx, dy, dz]], dtype=torch.float32)
kin = kin.to(device)
true_voxels = event['image'].reshape(1, 1, 5, 5, 5)
true_voxels = true_voxels.to(device)

# Run inference
with torch.no_grad():
    #generated_voxels = lit_model.sample(labels=kin, num_samples=1, num_timesteps=1000, device=device)
    generated_voxels = lit_model.model.differentiable_sample(kin=kin, num_timesteps=1000)

true_grid = true_voxels[0, 0].cpu().numpy()       # (5, 5, 5)
gen_grid  = generated_voxels[0, 0].cpu().numpy()  # (5, 5, 5)

# undo normalisation
min_charge = 0
max_charge = train_set.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['max']
true_grid = (true_grid + 1) / 2
true_grid = true_grid * (max_charge - min_charge) + min_charge
gen_grid = (gen_grid + 1) / 2
gen_grid = gen_grid * (max_charge - min_charge) + min_charge
gen_grid[gen_grid < 0.5] = 0  # zeroed voxels won't be plotted

global_max = float(max(true_grid.max(), gen_grid.max()))
cmin, cmax = 0.0, global_max if global_max > 0 else 1.0

def grid_to_sparse_points(grid, threshold=1e-6):
    """
    grid: numpy array (D, H, W)
    Returns arrays x, y, z, val for voxels above |value| > threshold.
    """
    import numpy as np

    mask = np.abs(grid) > threshold
    z_idx, y_idx, x_idx = mask.nonzero()  # (z, y, x)
    vals = grid[mask]
    return x_idx, y_idx, z_idx, vals


x_t, y_t, z_t, v_t = grid_to_sparse_points(true_grid)
x_g, y_g, z_g, v_g = grid_to_sparse_points(gen_grid)

def make_cube_traces(x, y, z, v, cmin, cmax, showscale_first=True):
    """
    Build a list of Mesh3d traces, one per (x, y, z) cube.
    Opacity is proportional to v/cmax.
    Each cube has hover text with its coordinates and value.
    """
    import numpy as np

    traces = []
    first = True

    for cx, cy, cz, val in zip(x, y, z, v):
        # 8 corners of cube [cx, cx+1] x [cy, cy+1] x [cz, cz+1]
        verts = np.array([
            (cx,   cy,   cz),
            (cx+1, cy,   cz),
            (cx+1, cy+1, cz),
            (cx,   cy+1, cz),
            (cx,   cy,   cz+1),
            (cx+1, cy,   cz+1),
            (cx+1, cy+1, cz+1),
            (cx,   cy+1, cz+1),
        ])
        xs, ys, zs = verts[:, 0], verts[:, 1], verts[:, 2]

        # 12 triangles (2 per face)
        faces = [
            (0, 1, 2), (0, 2, 3),  # bottom
            (4, 5, 6), (4, 6, 7),  # top
            (0, 1, 5), (0, 5, 4),  # front
            (2, 3, 7), (2, 7, 6),  # back
            (1, 2, 6), (1, 6, 5),  # right
            (0, 3, 7), (0, 7, 4),  # left
        ]
        i = [f[0] for f in faces]
        j = [f[1] for f in faces]
        k = [f[2] for f in faces]

        intensity = [val] * 8
        opacity = 0.0 if cmax <= 0 else float(val) / cmax

        hover_text = [f"x={cx}, y={cy}, z={cz}, value={val:.4g}"] * 8

        traces.append(
            go.Mesh3d(
                x=xs,
                y=ys,
                z=zs,
                i=i,
                j=j,
                k=k,
                intensity=intensity,
                intensitymode="vertex",
                colorscale="Viridis",
                cmin=cmin,
                cmax=cmax,
                opacity=opacity,
                flatshading=True,
                text=hover_text,
                hoverinfo="text",
                showscale=showscale_first and first,
                colorbar=dict(title="Energy") if (showscale_first and first) else None,
            )
        )
        if first:
            first = False

    return traces

true_traces = make_cube_traces(x_t, y_t, z_t, v_t, cmin, cmax, showscale_first=True)
gen_traces  = make_cube_traces(x_g, y_g, z_g, v_g, cmin, cmax, showscale_first=True)

# Plot
fig = make_subplots(
    rows=1,
    cols=2,
    specs=[[{"type": "scene"}, {"type": "scene"}]],
    subplot_titles=("True", "Generated"),
)
for tr in true_traces:
    fig.add_trace(tr, row=1, col=1)
for tr in gen_traces:
    fig.add_trace(tr, row=1, col=2)
axis_range = [-0.5, 5.5]
for col in [1, 2]:
    fig.update_scenes(
        xaxis=dict(title="x", range=axis_range),
        yaxis=dict(title="y", range=axis_range),
        zaxis=dict(title="z", range=axis_range),
        aspectmode="cube",
        row=1,
        col=col,
    )

fig.update_layout(
    showlegend=False,
)

fig.show()
